**Table of contents**<a id='toc0_'></a>    
- [Dataset Preparation and Batching](#toc1_)    
  - [Set Up the Dataset Environment](#toc1_1_)    
  - [Load the Processed Data](#toc1_2_)    
  - [Define the Translation Dataset](#toc1_3_)    
  - [Create the Train, Development, and Test Datasets](#toc1_4_)    
  - [Define Padding and Teacher-Forcing Shifts](#toc1_5_)    
  - [Create DataLoaders](#toc1_6_)    
  - [Inspect a Padded Batch](#toc1_7_)    
    - [Batch Shapes](#toc1_7_1_)    
  - [Demonstrate Sequence Padding](#toc1_8_)    
    - [Padded Sequence Example](#toc1_8_1_)    
  - [Demonstrate Decoder Input and Target Shifts](#toc1_9_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Dataset Preparation and Batching](#toc0_)

This notebook loads the processed translation pairs, wraps them in a PyTorch dataset, and prepares padded mini-batches for the Seq2Seq model.

## <a id='toc1_1_'></a>[Set Up the Dataset Environment](#toc0_)

Run this notebook with `experiments/` as the working directory.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pickle
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

## <a id='toc1_2_'></a>[Load the Processed Data](#toc0_)

In [2]:
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

with open(f'{DATA_DIR}/en_vocab.pkl', 'rb') as f:
    en_vocab = pickle.load(f)
with open(f'{DATA_DIR}/vi_vocab.pkl', 'rb') as f:
    vi_vocab = pickle.load(f)
with open(f'{DATA_DIR}/train_ids.pkl', 'rb') as f:
    train_ids = pickle.load(f)
with open(f'{DATA_DIR}/dev_ids.pkl', 'rb') as f:
    dev_ids = pickle.load(f)
with open(f'{DATA_DIR}/test_ids.pkl', 'rb') as f:
    test_ids = pickle.load(f)

print(f"Train: {len(train_ids)} pairs | Dev: {len(dev_ids)} | Test: {len(test_ids)}")

print(type(en_vocab))
print(type(vi_vocab))
print(type(train_ids))
print(type(dev_ids))
print(type(test_ids))

Train: 125798 pairs | Dev: 1511 | Test: 1192
<class 'src.vocab.Vocab'>
<class 'src.vocab.Vocab'>
<class 'list'>
<class 'list'>
<class 'list'>


## <a id='toc1_3_'></a>[Define the Translation Dataset](#toc0_)

In [3]:
class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs # list[(src_ids: list[int], tgt_ids: list[int])]
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)

## <a id='toc1_4_'></a>[Create the Train, Development, and Test Datasets](#toc0_)

In [4]:
train_dataset = TranslationDataset(train_ids) # list[(src_ids: list[int], tgt_ids: list[int])]
dev_dataset   = TranslationDataset(dev_ids)
test_dataset  = TranslationDataset(test_ids)

cnt = 0 
for x in train_dataset: # list[(src_ids: list[int], tgt_ids: list[int])]
    if cnt == 1: 
        break
    cnt += 1
    print(x[0])
    print(x[1])
    print()

tensor([    1,  6284, 15935,    58,     6,   297,   555,    12,   684,  5279,
            2])
tensor([   1,  293,  912,   84,    9, 2836,   33,  831,    2])



## <a id='toc1_5_'></a>[Define Padding and Teacher-Forcing Shifts](#toc0_)

In [5]:
PAD_IDX = 0 # đã định nghĩa trong Vocab: {'<pad>': 0, ...}
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch) # tách ra thành src_batch và tgt_batch
    src_lengths = torch.tensor([len(s) for s in src_batch]) # độ dài của từng smaple trong từng batch
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)
    # tgt_padded: [<bos>, i, am, a, student, <eos>]
    # tgt_input  = [<bos>, i, am, a, student] bỏ token CUỐI
    # tgt_output = [i, am, a, student, <eos>] bỏ token ĐẦU
    tgt_input  = tgt_padded[:, :-1]   # INPUT cho decoder
    tgt_output = tgt_padded[:, 1:]    # TARGET để tính loss
    return src_padded, src_lengths, tgt_input, tgt_output


## <a id='toc1_6_'></a>[Create DataLoaders](#toc0_)

In [6]:
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(type(train_loader))

<class 'torch.utils.data.dataloader.DataLoader'>


In [7]:
print(len(train_dataset))
print(len(train_loader))

125798
1966


## <a id='toc1_7_'></a>[Inspect a Padded Batch](#toc0_)

The next cell inspects the shapes produced by the collate function.

- `src_batch` contains padded English source sequences for the encoder.
- `src_lens` stores the original source lengths before padding.

### <a id='toc1_7_1_'></a>[Batch Shapes](#toc0_)

In [8]:
src_batch, src_lens, tgt_in, tgt_out = next(iter(train_loader)) # lấy 1 batch ra.

print("src_batch shape:", src_batch.shape)
print("tgt_input shape:", tgt_in.shape) 
print("tgt_output shape:", tgt_out.shape)



src_batch shape: torch.Size([64, 50])
tgt_input shape: torch.Size([64, 48])
tgt_output shape: torch.Size([64, 48])


## <a id='toc1_8_'></a>[Demonstrate Sequence Padding](#toc0_)

### <a id='toc1_8_1_'></a>[Padded Sequence Example](#toc0_)

In [9]:
seq1 = torch.tensor([1, 2, 3])
seq2 = torch.tensor([2, 3])
seq3 = torch.tensor([1])
padded_sequences = pad_sequence([seq1, seq2, seq3], batch_first=True, padding_value=0)
print(padded_sequences)

tensor([[1, 2, 3],
        [2, 3, 0],
        [1, 0, 0]])


## <a id='toc1_9_'></a>[Demonstrate Decoder Input and Target Shifts](#toc0_)

In [10]:
seq1 = torch.tensor([1, 2, 3, 4, 5, 6])
print(seq1[:-1])
print(seq1[1:])

tensor([1, 2, 3, 4, 5])
tensor([2, 3, 4, 5, 6])


In [ ]:
from pathlib import Path
from src.data import load_corpus, make_loaders
corpus = load_corpus(Path("../data"))
train_loader, dev_loader, test_loader = make_loaders(corpus, batch_size=64)

src_padded, src_lengths, tgt_input, tgt_output = next(iter(train_loader))

print("Number of batches:", len(train_loader), len(dev_loader), len(test_loader))
print("Batch shapes:")
print("src_padded:", src_padded.shape)
print("src_lengths:", src_lengths.shape)
print("tgt_input:", tgt_input.shape)
print("tgt_output:", tgt_output.shape)
print("First source length:", src_lengths[0].item())
print("First target input:", tgt_input[0, :10].tolist())
print("First target output:", tgt_output[0, :10].tolist())